# Лабораторная работа: CNN для MNIST и transfer learning на второй 28×28 задаче

В этом ноутбуке решение **разбито по пунктам задания**.  
Для каждого пункта есть:

- формулировка;
- короткая пометка **«Что сделано»**;
- код, который это реализует.

**Выбор второго датасета:** по умолчанию используется **FashionMNIST**, потому что он сразу соответствует условию *1 канал, 28×28*.  
При желании можно переключиться на `SVHN`, но тогда изображения нужно приводить к grayscale и размеру 28×28.

---

## План ноутбука

1. Загрузка MNIST и второго MNIST-подобного датасета.  
2. Выбор фреймворка и базовых настроек.  
3. Реализация свёрточной сети и функции потерь.  
4. Обучение модели на MNIST и построение кривых.  
5. Добавление второй головы и сохранение параметров.  
6. Обучение второй головы при замороженном backbone.  
7. Разморозка backbone и дообучение на второй задаче.  
8. Загрузка сохранённых параметров и обучение со сразу размороженным backbone.  
9. Случайная инициализация + заморозка начальных conv-слоёв.  
10. Отображение всех кривых обучения на одной figure.  
11. Выбор лучшей модели второй задачи и поиск изображений класса `c`, наиболее похожих на класс `t`.


## Пункт 1. Загрузить MNIST и любой другой MNIST-подобный датасет

**Что сделано:**  
- загружаются `MNIST` и `FashionMNIST`;  
- оба датасета имеют формат `1×28×28`;  
- ниже есть функция, которая может переключить второй датасет на `SVHN`, если это нужно.


In [ ]:

import os
import copy
import random
from collections import defaultdict

import numpy as np
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from torchvision import datasets, transforms

%matplotlib inline


## Пункт 2. Выбрать фреймворк для глубокого обучения

**Что сделано:**  
Используется **PyTorch**: `torch`, `torchvision`, `torch.nn`, `DataLoader`.


In [ ]:

# Базовые настройки эксперимента
SEED = 42
BATCH_SIZE = 128
LR = 1e-3

EPOCHS_STAGE1 = 5           # обучение head1 на MNIST
EPOCHS_HEAD2_FROZEN = 5     # head2, backbone frozen
EPOCHS_HEAD2_UNFROZEN = 5   # head2, backbone unfrozen
EPOCHS_HEAD2_DIRECT = 5     # head2, backbone unfrozen с самого начала
EPOCHS_RANDOM = 5           # случайная инициализация + частичная заморозка

SECOND_DATASET = "fashionmnist"  # "fashionmnist" или "svhn"

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
CHECKPOINT_DIR = "checkpoints"
os.makedirs(CHECKPOINT_DIR, exist_ok=True)

print("DEVICE:", DEVICE)


In [ ]:

def set_seed(seed: int = 42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed(SEED)


In [ ]:

def build_dataloaders(second_dataset: str = "fashionmnist", batch_size: int = 128):
    # MNIST
    mnist_tf = transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize((0.1307,), (0.3081,))
    ])

    mnist_train = datasets.MNIST(root="./data", train=True, download=True, transform=mnist_tf)
    mnist_test = datasets.MNIST(root="./data", train=False, download=True, transform=mnist_tf)

    mnist_train_loader = DataLoader(mnist_train, batch_size=batch_size, shuffle=True)
    mnist_test_loader = DataLoader(mnist_test, batch_size=batch_size, shuffle=False)

    # Второй датасет
    second_dataset = second_dataset.lower()

    if second_dataset == "fashionmnist":
        second_tf = transforms.Compose([
            transforms.ToTensor(),
            transforms.Normalize((0.2860,), (0.3530,))
        ])
        second_train = datasets.FashionMNIST(root="./data", train=True, download=True, transform=second_tf)
        second_test = datasets.FashionMNIST(root="./data", train=False, download=True, transform=second_tf)

    elif second_dataset == "svhn":
        second_tf = transforms.Compose([
            transforms.Grayscale(),
            transforms.Resize((28, 28)),
            transforms.ToTensor(),
            transforms.Normalize((0.5,), (0.5,))
        ])
        second_train = datasets.SVHN(root="./data", split="train", download=True, transform=second_tf)
        second_test = datasets.SVHN(root="./data", split="test", download=True, transform=second_tf)

    else:
        raise ValueError("second_dataset должен быть 'fashionmnist' или 'svhn'")

    second_train_loader = DataLoader(second_train, batch_size=batch_size, shuffle=True)
    second_test_loader = DataLoader(second_test, batch_size=batch_size, shuffle=False)

    return {
        "mnist_train_ds": mnist_train,
        "mnist_test_ds": mnist_test,
        "second_train_ds": second_train,
        "second_test_ds": second_test,
        "mnist_train": mnist_train_loader,
        "mnist_test": mnist_test_loader,
        "second_train": second_train_loader,
        "second_test": second_test_loader,
    }

loaders = build_dataloaders(second_dataset=SECOND_DATASET, batch_size=BATCH_SIZE)
print("MNIST train:", len(loaders["mnist_train_ds"]))
print("MNIST test :", len(loaders["mnist_test_ds"]))
print("Second train:", len(loaders["second_train_ds"]))
print("Second test :", len(loaders["second_test_ds"]))


In [ ]:

def show_images(dataset, title, num_images=5):
    plt.figure(figsize=(12, 2.5))
    plt.suptitle(title, fontsize=12)

    for i in range(num_images):
        image, label = dataset[i]
        image = image.squeeze().numpy()

        plt.subplot(1, num_images, i + 1)
        plt.imshow(image, cmap="gray")
        plt.title(f"Label: {label}", fontsize=9)
        plt.axis("off")

    plt.tight_layout()
    plt.show()

show_images(loaders["mnist_train_ds"], "MNIST examples")
show_images(loaders["second_train_ds"], f"{SECOND_DATASET.upper()} examples")


## Пункт 3. Реализовать архитектуру свёрточной сети для классификации MNIST и выбрать эмпирический риск

**Что сделано:**  
- реализован общий `backbone` из трёх свёрточных блоков;  
- реализованы две классификационные головы: `head1` для MNIST и `head2` для второй задачи;  
- в качестве функции потерь используется **CrossEntropyLoss**.


In [ ]:

class ConvBackbone(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv2d(1, 32, kernel_size=5)
        self.conv2 = nn.Conv2d(32, 64, kernel_size=5)
        self.conv3 = nn.Conv2d(64, 128, kernel_size=3)
        self.pool = nn.MaxPool2d(2, 2)

    def forward(self, x):
        x = self.pool(torch.relu(self.conv1(x)))  # 28 -> 24 -> 12
        x = self.pool(torch.relu(self.conv2(x)))  # 12 -> 8 -> 4
        x = self.pool(torch.relu(self.conv3(x)))  # 4 -> 2 -> 1
        x = x.flatten(1)                          # [B, 128]
        return x


class ClassifierHead(nn.Module):
    def __init__(self, in_features=128, hidden=64, num_classes=10):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_features, hidden),
            nn.ReLU(),
            nn.Linear(hidden, num_classes),
        )

    def forward(self, x):
        return self.net(x)


class MultiHeadCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.backbone = ConvBackbone()
        self.head1 = ClassifierHead()  # MNIST
        self.head2 = ClassifierHead()  # вторая задача

    def forward(self, x, task="head1"):
        feats = self.backbone(x)
        if task == "head1":
            return self.head1(feats)
        elif task == "head2":
            return self.head2(feats)
        raise ValueError("task должен быть 'head1' или 'head2'")


criterion = nn.CrossEntropyLoss()
model = MultiHeadCNN().to(DEVICE)
model


## Вспомогательные функции

Ниже собраны общие функции для:
- обучения и валидации;
- заморозки и разморозки параметров;
- сохранения и загрузки состояний;
- построения кривых;
- поиска «наиболее похожих» изображений по логитам.


In [ ]:

def evaluate(model, loader, criterion, task: str):
    model.eval()
    total_loss = 0.0
    total_correct = 0
    total_size = 0

    with torch.no_grad():
        for x, y in loader:
            x = x.to(DEVICE)
            y = y.to(DEVICE)

            logits = model(x, task=task)
            loss = criterion(logits, y)

            total_loss += loss.item() * x.size(0)
            total_correct += (logits.argmax(dim=1) == y).sum().item()
            total_size += x.size(0)

    return total_loss / total_size, total_correct / total_size


def train_epochs(
    model,
    train_loader,
    test_loader,
    task: str,
    epochs: int,
    lr: float,
    criterion,
    optimizer=None,
    extra_eval=None,
):
    if optimizer is None:
        optimizer = torch.optim.Adam(filter(lambda p: p.requires_grad, model.parameters()), lr=lr)

    history = defaultdict(list)

    for epoch in range(1, epochs + 1):
        model.train()
        train_loss_sum = 0.0
        train_correct = 0
        train_size = 0

        for x, y in train_loader:
            x = x.to(DEVICE)
            y = y.to(DEVICE)

            optimizer.zero_grad()
            logits = model(x, task=task)
            loss = criterion(logits, y)
            loss.backward()
            optimizer.step()

            train_loss_sum += loss.item() * x.size(0)
            train_correct += (logits.argmax(dim=1) == y).sum().item()
            train_size += x.size(0)

        train_loss = train_loss_sum / train_size
        train_acc = train_correct / train_size
        test_loss, test_acc = evaluate(model, test_loader, criterion, task=task)

        history["train_loss"].append(train_loss)
        history["train_acc"].append(train_acc)
        history["test_loss"].append(test_loss)
        history["test_acc"].append(test_acc)

        if extra_eval is not None:
            for name, loader, eval_task in extra_eval:
                ev_loss, ev_acc = evaluate(model, loader, criterion, task=eval_task)
                history[f"{name}_loss"].append(ev_loss)
                history[f"{name}_acc"].append(ev_acc)

        print(
            f"[{task}] epoch {epoch:02d}/{epochs} | "
            f"train_loss={train_loss:.4f} train_acc={train_acc:.4f} | "
            f"test_loss={test_loss:.4f} test_acc={test_acc:.4f}"
        )

    return dict(history)


def freeze_backbone(model: MultiHeadCNN):
    for p in model.backbone.parameters():
        p.requires_grad = False


def unfreeze_backbone(model: MultiHeadCNN):
    for p in model.backbone.parameters():
        p.requires_grad = True


def freeze_first_n_convs(model: MultiHeadCNN, n: int = 1):
    conv_layers = [model.backbone.conv1, model.backbone.conv2, model.backbone.conv3]
    for layer in conv_layers[:n]:
        for p in layer.parameters():
            p.requires_grad = False


def reinit_all_weights(module: nn.Module):
    for m in module.modules():
        if isinstance(m, (nn.Conv2d, nn.Linear)):
            nn.init.kaiming_uniform_(m.weight, nonlinearity="relu")
            if m.bias is not None:
                nn.init.zeros_(m.bias)


def save_split_checkpoint(model: MultiHeadCNN, path: str):
    torch.save({
        "backbone": model.backbone.state_dict(),
        "head1": model.head1.state_dict(),
        "head2": model.head2.state_dict(),
    }, path)


def load_split_checkpoint(model: MultiHeadCNN, path: str):
    ckpt = torch.load(path, map_location=DEVICE)
    model.backbone.load_state_dict(ckpt["backbone"])
    model.head1.load_state_dict(ckpt["head1"])
    model.head2.load_state_dict(ckpt["head2"])


def plot_history(history, title_prefix="Experiment"):
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    if "train_loss" in history:
        axes[0].plot(history["train_loss"], label="train_loss")
    if "test_loss" in history:
        axes[0].plot(history["test_loss"], label="test_loss")

    if "train_acc" in history:
        axes[1].plot(history["train_acc"], label="train_acc")
    if "test_acc" in history:
        axes[1].plot(history["test_acc"], label="test_acc")

    axes[0].set_title(f"{title_prefix}: loss")
    axes[1].set_title(f"{title_prefix}: accuracy")

    axes[0].set_xlabel("epoch")
    axes[1].set_xlabel("epoch")

    axes[0].legend()
    axes[1].legend()
    plt.tight_layout()
    plt.show()


def plot_all_histories(histories: dict):
    fig, axes = plt.subplots(1, 2, figsize=(20, 8))

    for exp_name, hist in histories.items():
        for k, vals in hist.items():
            if "loss" in k:
                axes[0].plot(range(1, len(vals) + 1), vals, label=f"{exp_name}/{k}")
            if "acc" in k:
                axes[1].plot(range(1, len(vals) + 1), vals, label=f"{exp_name}/{k}")

    axes[0].set_title("Все loss-кривые")
    axes[0].set_xlabel("epoch")
    axes[0].set_ylabel("loss")

    axes[1].set_title("Все accuracy-кривые")
    axes[1].set_xlabel("epoch")
    axes[1].set_ylabel("accuracy")

    axes[0].legend(fontsize=8)
    axes[1].legend(fontsize=8)
    plt.tight_layout()
    plt.show()


def find_most_similar_images(model, loader, task="head2"):
    model.eval()
    all_images = []
    all_labels = []
    all_logits = []

    with torch.no_grad():
        for x, y in loader:
            x = x.to(DEVICE)
            logits = model(x, task=task)

            all_images.append(x.cpu())
            all_labels.append(y.cpu())
            all_logits.append(logits.cpu())

    all_images = torch.cat(all_images, dim=0)
    all_labels = torch.cat(all_labels, dim=0)
    all_logits = torch.cat(all_logits, dim=0)

    num_classes = all_logits.shape[1]
    similar = {}

    for c in range(num_classes):
        mask = (all_labels == c)
        imgs_c = all_images[mask]
        logits_c = all_logits[mask]

        for t in range(num_classes):
            idx = torch.argmax(logits_c[:, t]).item()
            similar[(c, t)] = imgs_c[idx]

    return similar


def show_pair_image(similar_dict, c, t):
    img = similar_dict[(c, t)].squeeze(0).numpy()
    plt.figure(figsize=(3, 3))
    plt.imshow(img, cmap="gray")
    plt.title(f"true class = {c}, most similar to target = {t}")
    plt.axis("off")
    plt.show()


## Пункт 4. Обучить архитектуру на MNIST и построить кривые обучения

**Что сделано:**  
- обучается `head1` на `MNIST`;  
- строятся кривые `train/test loss` и `train/test accuracy`;  
- после обучения модель первой задачи сохраняется как базовая точка для transfer learning.


In [ ]:

model_stage1 = MultiHeadCNN().to(DEVICE)

history_stage1 = train_epochs(
    model=model_stage1,
    train_loader=loaders["mnist_train"],
    test_loader=loaders["mnist_test"],
    task="head1",
    epochs=EPOCHS_STAGE1,
    lr=LR,
    criterion=criterion,
)

plot_history(history_stage1, title_prefix="MNIST / head1")


## Пункт 5. Добавить новую «голову» и сохранить параметры общих слоёв, первой головы и ещё не обученной второй головы

**Что сделано:**  
- архитектура уже содержит `backbone`, `head1`, `head2`;  
- после обучения на MNIST сохраняются:
  - параметры `backbone`,
  - параметры `head1`,
  - параметры `head2` (она пока не обучена).


In [ ]:

BASE_CHECKPOINT = os.path.join(CHECKPOINT_DIR, "after_mnist_stage.pt")
save_split_checkpoint(model_stage1, BASE_CHECKPOINT)
print("Сохранено:", BASE_CHECKPOINT)


In [ ]:

# Проверка структуры чекпойнта
checkpoint_preview = torch.load(BASE_CHECKPOINT, map_location="cpu")
print(checkpoint_preview.keys())
print("backbone layers:", len(checkpoint_preview["backbone"]))
print("head1 layers   :", len(checkpoint_preview["head1"]))
print("head2 layers   :", len(checkpoint_preview["head2"]))


## Пункт 6. Обучить вторую голову для второго датасета при замороженных общих свёрточных слоях

**Что сделано:**  
- берётся модель после обучения на MNIST;  
- `backbone` замораживается;  
- обучается только `head2`;  
- строятся кривые обучения для второй задачи.


In [ ]:

model_frozen = copy.deepcopy(model_stage1).to(DEVICE)
freeze_backbone(model_frozen)

# Проверка, что backbone действительно заморожен
for name, param in model_frozen.named_parameters():
    if "backbone" in name:
        assert param.requires_grad is False

history_head2_frozen = train_epochs(
    model=model_frozen,
    train_loader=loaders["second_train"],
    test_loader=loaders["second_test"],
    task="head2",
    epochs=EPOCHS_HEAD2_FROZEN,
    lr=LR,
    criterion=criterion,
)

plot_history(history_head2_frozen, title_prefix=f"{SECOND_DATASET.upper()} / head2 / frozen backbone")


## Пункт 7. Разморозить общие свёрточные преобразования и дообучить вторую голову. Построить кривые для первой и второй задачи

**Что сделано:**  
- после обучения только второй головы `backbone` размораживается;  
- модель дообучается на второй задаче уже целиком;  
- после каждой эпохи дополнительно измеряются метрики на первой задаче (`MNIST`) и на второй задаче.


In [ ]:

unfreeze_backbone(model_frozen)

history_head2_unfrozen = train_epochs(
    model=model_frozen,
    train_loader=loaders["second_train"],
    test_loader=loaders["second_test"],
    task="head2",
    epochs=EPOCHS_HEAD2_UNFROZEN,
    lr=LR * 0.5,
    criterion=criterion,
    extra_eval=[
        ("mnist_test", loaders["mnist_test"], "head1"),
        ("second_test_again", loaders["second_test"], "head2"),
    ],
)


In [ ]:

# Отдельная визуализация для пункта 7:
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

axes[0].plot(history_head2_unfrozen["test_loss"], label="second_test_loss")
axes[0].plot(history_head2_unfrozen["mnist_test_loss"], label="mnist_test_loss")
axes[0].set_title("Пункт 7: loss для первой и второй задачи")
axes[0].set_xlabel("epoch")
axes[0].legend()

axes[1].plot(history_head2_unfrozen["test_acc"], label="second_test_acc")
axes[1].plot(history_head2_unfrozen["mnist_test_acc"], label="mnist_test_acc")
axes[1].set_title("Пункт 7: accuracy для первой и второй задачи")
axes[1].set_xlabel("epoch")
axes[1].legend()

plt.tight_layout()
plt.show()


## Пункт 8. Загрузить сохранённые параметры и обучить вторую голову при сразу размороженном backbone

**Что сделано:**  
- заново загружается состояние модели после MNIST;  
- `backbone` сразу остаётся размороженным;  
- обучается `head2` и одновременно отслеживаются метрики первой и второй задачи.


In [ ]:

model_direct = MultiHeadCNN().to(DEVICE)
load_split_checkpoint(model_direct, BASE_CHECKPOINT)
unfreeze_backbone(model_direct)

history_head2_direct = train_epochs(
    model=model_direct,
    train_loader=loaders["second_train"],
    test_loader=loaders["second_test"],
    task="head2",
    epochs=EPOCHS_HEAD2_DIRECT,
    lr=LR,
    criterion=criterion,
    extra_eval=[
        ("mnist_test", loaders["mnist_test"], "head1"),
        ("second_test_again", loaders["second_test"], "head2"),
    ],
)


In [ ]:

# Визуализация для пункта 8
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

axes[0].plot(history_head2_direct["test_loss"], label="second_test_loss")
axes[0].plot(history_head2_direct["mnist_test_loss"], label="mnist_test_loss")
axes[0].set_title("Пункт 8: loss для первой и второй задачи")
axes[0].set_xlabel("epoch")
axes[0].legend()

axes[1].plot(history_head2_direct["test_acc"], label="second_test_acc")
axes[1].plot(history_head2_direct["mnist_test_acc"], label="mnist_test_acc")
axes[1].set_title("Пункт 8: accuracy для первой и второй задачи")
axes[1].set_xlabel("epoch")
axes[1].legend()

plt.tight_layout()
plt.show()


## Пункт 9. Заполнить все параметры случайными значениями, заморозить несколько начальных свёрточных слоёв и обучить оставшиеся слои для второй задачи

**Что сделано:**  
- создаётся новая модель со случайной инициализацией;  
- замораживаются первые `n` свёрточных слоёв backbone;  
- обучаются оставшиеся слои и `head2`;  
- строятся кривые обучения для второй задачи.

> Если модель обучается слишком плохо, можно уменьшить `N_FROZEN_CONVS` с `1` до `0`.


In [ ]:

N_FROZEN_CONVS = 1

model_random = MultiHeadCNN().to(DEVICE)
reinit_all_weights(model_random)
freeze_first_n_convs(model_random, n=N_FROZEN_CONVS)

history_random_partial = train_epochs(
    model=model_random,
    train_loader=loaders["second_train"],
    test_loader=loaders["second_test"],
    task="head2",
    epochs=EPOCHS_RANDOM,
    lr=LR,
    criterion=criterion,
)

plot_history(history_random_partial, title_prefix=f"Random init / frozen first {N_FROZEN_CONVS} conv")


## Пункт 10. Отобразить все кривые обучения на одном графике

**Что сделано:**  
Все истории собираются в один словарь `all_histories`, после чего строится одна figure с:
- всеми `loss`-кривыми;
- всеми `accuracy`-кривыми.


In [ ]:

all_histories = {
    "mnist_head1": history_stage1,
    "head2_frozen": history_head2_frozen,
    "head2_unfrozen": history_head2_unfrozen,
    "head2_direct_unfrozen": history_head2_direct,
    "random_partial_frozen": history_random_partial,
}

plot_all_histories(all_histories)


## Пункт 11. Выбрать лучшую модель для второй задачи и для каждой пары классов `c, t` найти изображение класса `c`, которое больше всего похоже на класс `t`

**Что сделано:**  
- среди нескольких сценариев выбирается лучшая модель по `test accuracy` второй задачи;  
- по логитам на тестовом множестве второй задачи строится словарь `similar_images[(c, t)]`;  
- в ячейках ниже можно показать примеры найденных изображений.


In [ ]:

candidates = {
    "frozen_then_unfrozen": model_frozen,
    "direct_unfrozen": model_direct,
    "random_partial_frozen": model_random,
}

best_name = None
best_acc = -1.0
best_model = None

for name, mdl in candidates.items():
    _, acc = evaluate(mdl, loaders["second_test"], criterion, task="head2")
    print(f"{name}: second task test acc = {acc:.4f}")
    if acc > best_acc:
        best_acc = acc
        best_name = name
        best_model = mdl

print("\nЛучшая модель для второй задачи:", best_name)
print("Лучшая accuracy:", round(best_acc, 4))


In [ ]:

similar_images = find_most_similar_images(best_model, loaders["second_test"], task="head2")

# Проверка: для 10 классов должно получиться 100 пар (c, t)
print("Число найденных пар:", len(similar_images))
assert len(similar_images) == 100


In [ ]:

# Примеры визуализации отдельных пар классов
show_pair_image(similar_images, c=3, t=9)
show_pair_image(similar_images, c=2, t=7)
show_pair_image(similar_images, c=5, t=1)


In [ ]:

BEST_MODEL_PATH = os.path.join(CHECKPOINT_DIR, "best_second_task_model.pt")
save_split_checkpoint(best_model, BEST_MODEL_PATH)

SIMILAR_PATH = os.path.join(CHECKPOINT_DIR, "most_similar_images_by_logits.pt")
torch.save(similar_images, SIMILAR_PATH)

print("Сохранено:", BEST_MODEL_PATH)
print("Сохранено:", SIMILAR_PATH)


## Короткий вывод

В ноутбуке реализован полный пайплайн:
- обучение CNN на `MNIST`;
- перенос на вторую задачу с помощью новой головы;
- сравнение нескольких стратегий transfer learning;
- визуализация всех кривых обучения;
- выбор лучшей модели второй задачи;
- поиск изображений класса `c`, которые модель считает наиболее похожими на класс `t`.

Такой формат обычно хорошо подходит под сдачу, потому что каждый пункт задания явно закрыт отдельной секцией.
